# تدريب نموذج Ayso 3D على كارت مجاني (Google Colab T4)### نص ← شبكة إشغال 32³ بنموذج التفافي — بدون فلوس**التشغيل:**1. `Settings → Accelerator → T4 GPU`2. (ما يحتاج — النت مفتوح أصلاً)3. `Run All`**الزمن:** ~٢٥ دقيقة أول مرة · **الحصة:** ~٢٠ ساعة/أسبوع مجاناً · **بلا كارت بنكي وبلا تحقق هاتف****المخرجات:** `ayso3d-model.zip` — ينزل تلقائياً على جهازك

In [ ]:
# ١) المكتبات!pip -q install trimesh scikit-image 2>/dev/null | tail -1import torchprint("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ٢) تنزيل ModelNet40 (رخصة MIT) — ١٢ مجسّم لكل صنفimport osfrom huggingface_hub import snapshot_downloadCLASSES = ["airplane","bathtub","bed","bench","bookshelf","bottle","bowl","car","chair","cone",           "cup","curtain","desk","door","dresser","flower_pot","glass_box","guitar","keyboard","lamp",           "laptop","mantel","monitor","night_stand","person","piano","plant","radio","range_hood","sink",           "sofa","stairs","stool","table","tent","toilet","tv_stand","vase","wardrobe","xbox"]DATA = "/content/data" if os.path.isdir("/content") else "./data"os.makedirs(DATA, exist_ok=True)patterns = [f"ModelNet40/{c}/train/*.off" for c in CLASSES]path = snapshot_download("naderalfares/ModelNet40", repo_type="dataset",                         local_dir=DATA, allow_patterns=patterns)print("تم التنزيل:", path)

In [ ]:
# ٣) الفوكسلاز: رؤوس + تغطية مثلثات + إغلاق + حشو الداخل ← 32³import numpy as npdef read_off(path):    lines = open(path, 'rb').read().decode('latin1').split('\n')    p = 0    while p < len(lines) and 'OFF' not in lines[p]: p += 1    head = lines[p].replace('OFF','').strip().split()    if len(head) >= 2: nv, nf = int(head[0]), int(head[1]); p += 1    else: p += 1; h = lines[p].split(); nv, nf = int(h[0]), int(h[1]); p += 1    V = np.array(' '.join(lines[p:p+nv]).split(), dtype=np.float32).reshape(nv, 3); p += nv    body = ' '.join(lines[p:p+nf]).split()    F, i = [], 0    while i < len(body):        m = int(body[i])        if m == 3: F.append((int(body[i+1]), int(body[i+2]), int(body[i+3]))); i += 4        elif m == 4:            F.append((int(body[i+1]), int(body[i+2]), int(body[i+3])))            F.append((int(body[i+1]), int(body[i+3]), int(body[i+4]))); i += 5        else: i += 1 + m    return V, np.array(F, dtype=np.int32)def voxelize(V, F, N=32, tri_budget=12000):    lo, hi = np.percentile(V, 0.5, axis=0), np.percentile(V, 99.5, axis=0)    keep = np.all((V >= lo) & (V <= hi), axis=1)    if keep.sum() > 100: V = V[keep]    mn, mx = V.min(0), V.max(0)    span = float((mx - mn).max()) or 1.0    cX, cY = (mn[0]+mx[0])/2, (mn[1]+mx[1])/2    G = np.empty_like(V)    G[:,0] = (V[:,0]-cX)/span*N + N/2    G[:,1] = (V[:,2]-mn[2])/span*N    G[:,2] = (V[:,1]-cY)/span*N + N/2    occ = np.zeros((N,N,N), bool)    iv = np.floor(G).astype(int); ok = np.all((iv>=0)&(iv<N), axis=1)    occ[iv[ok,0], iv[ok,1], iv[ok,2]] = True    if len(F) > tri_budget: F = F[np.random.choice(len(F), tri_budget, replace=False)]    a, b, c = G[F[:,0]], G[F[:,1]], G[F[:,2]]    e = np.max(np.stack([np.linalg.norm(b-a,axis=1), np.linalg.norm(c-b,axis=1),                         np.linalg.norm(a-c,axis=1)]), axis=0)    S = np.clip(np.ceil(e*1.5), 2, 14).astype(int)    for k in range(len(F)):        s = S[k]; u = np.arange(s+1); uu, vv = np.meshgrid(u, u, indexing='ij')        m = (uu+vv) <= s        t1 = uu[m]/s; t2 = vv[m]/s; t0 = 1-t1-t2        pts = a[k]*t0[:,None] + b[k]*t1[:,None] + c[k]*t2[:,None]        ip = np.floor(pts).astype(int); ok = np.all((ip>=0)&(ip<N), axis=1)        occ[ip[ok,0], ip[ok,1], ip[ok,2]] = True    for _ in range(2):        pad = np.pad(occ, 1)        cnt = sum(pad[1+dx:1+dx+N, 1+dy:1+dy+N, 1+dz:1+dz+N]                  for dx in (-1,0,1) for dy in (-1,0,1) for dz in (-1,0,1) if (dx,dy,dz)!=(0,0,0))        occ |= cnt >= 7    air = np.zeros_like(occ)    air[0,:,:] = air[-1,:,:] = air[:,0,:] = air[:,-1,:] = air[:,:,0] = air[:,:,-1] = True    air &= ~occ    while True:        pad = np.pad(air, 1)        grow = (pad[0:N,1:N+1,1:N+1] | pad[2:N+2,1:N+1,1:N+1] |                pad[1:N+1,0:N,1:N+1] | pad[1:N+1,2:N+2,1:N+1] |                pad[1:N+1,1:N+1,0:N] | pad[1:N+1,1:N+1,2:N+2]) & ~occ        new = air | grow        if np.array_equal(new, air): break        air = new    return ~airimport globf = sorted(glob.glob(DATA + "/ModelNet40/chair/train/*.off"))[0]V, Fc = read_off(f); O = voxelize(V, Fc, 32)print("اختبار:", f.split('/')[-1], "| امتلاء", round(O.mean()*100,1), "%")m = O.max(axis=2)for j in range(31, -1, -1):    print(''.join('#' if m[i,j] else '.' for i in range(32)))

In [ ]:
# ٤) بناء مجموعة البيانات (تُحفظ مرة واحدة)import os, glob, numpy as npN = 32CACHE = ("/content" if os.path.isdir("/content") else ".") + "/vox32.npz"if os.path.exists(CACHE):    d = np.load(CACHE, allow_pickle=True)    occs, cls_names = d['occs'], d['cls']    print("محفوظة مسبقاً:", len(occs))else:    L, K = [], []    for c in CLASSES:        files = sorted(glob.glob(f"{DATA}/ModelNet40/{c}/train/*.off"))[:12]        for f in files:            try:                V, F = read_off(f)                if len(V) < 20 or len(F) < 10: continue                L.append(voxelize(V, F, N)); K.append(c)            except Exception:                pass        print(f"{c:14s} {len([x for x in K if x==c]):2d} ✓", flush=True)    occs = np.array(L, dtype=bool); cls_names = np.array(K)    np.savez_compressed(CACHE, occs=occs, cls=cls_names)    print("حُفظت في", CACHE)print("المجموع:", occs.shape, "| متوسط الامتلاء:", round(occs.mean()*100, 2), "%")

In [ ]:
# ٥) الموديل: نص ← متجه ← فك تشفير التفافي 4³ → 32³import torch, torch.nn as nnimport numpy as npD_TXT = 2048def text_feat(text, D=D_TXT):    # تجزئة كلمات + ثلاثيات حروف لكل كلمة على حدة (تشابه أعلى بين الصيغ)    acc = np.zeros(D, np.float32)    def h(s):        x = 2166136261        for ch in s.encode('utf-8'):            x ^= ch; x = (x * 16777619) & 0xFFFFFFFF        return x    for w in text.strip().lower().split():        acc[h("w:" + w) % D] += 2.2        t = "«" + w + "»"        for i in range(len(t) - 2):            acc[h("3:" + t[i:i+3]) % D] += 1.0    n = np.linalg.norm(acc)    return acc / n if n > 0 else accclass Gen(nn.Module):    def __init__(self, N=32, cond=256, base=128):        super().__init__()        self.enc = nn.Sequential(nn.Linear(D_TXT, 512), nn.ReLU(), nn.Linear(512, cond), nn.ReLU())        self.fc = nn.Linear(cond, base * 4 * 4 * 4)        self.up1 = nn.Sequential(nn.ConvTranspose3d(base, 96, 4, 2, 1), nn.BatchNorm3d(96), nn.ReLU())        self.up2 = nn.Sequential(nn.ConvTranspose3d(96, 64, 4, 2, 1), nn.BatchNorm3d(64), nn.ReLU())        self.up3 = nn.Sequential(nn.ConvTranspose3d(64, 32, 4, 2, 1), nn.BatchNorm3d(32), nn.ReLU())        self.out = nn.Conv3d(32, 1, 1)    def forward(self, x):        h = self.enc(x)        h = self.fc(h).view(-1, 128, 4, 4, 4)        h = self.up1(h); h = self.up2(h); h = self.up3(h)        return self.out(h).squeeze(1)NAMES = { "airplane":["airplane","طائرة","jet plane","طائرة كبيرة"], "bathtub":["bathtub","حوض استحمام","bath tub","حوض"], "bed":["bed","سرير","wooden bed","سرير خشبي"], "bench":["bench","مقعد","park bench","مقعد حديقة"], "bookshelf":["bookshelf","رف كتب","book shelf","مكتبة"], "bottle":["bottle","قنينة","glass bottle","زجاجة"], "bowl":["bowl","وعاء","deep bowl","صحن"], "car":["car","سيارة","sedan car","سيارة حمراء"], "chair":["chair","كرسي","wooden chair","كرسي خشبي"], "cone":["cone","مخروط","traffic cone","قمع"], "cup":["cup","كأس","coffee cup","كوب"], "curtain":["curtain","ستارة","window curtain","ستائر"], "desk":["desk","مكتب","office desk","طاولة مكتب"], "door":["door","باب","wooden door","باب خشبي"], "dresser":["dresser","خزانة","drawer dresser","خزانة ملابس"], "flower_pot":["flower pot","اصيص","plant pot","اصيص زهور"], "glass_box":["glass box","صندوق زجاجي","glass case","علبة زجاج"], "guitar":["guitar","قيثارة","acoustic guitar","جيتار"], "keyboard":["keyboard","لوحة مفاتيح","computer keyboard","كيبورد"], "lamp":["lamp","مصباح","table lamp","مصباح طاولة"], "laptop":["laptop","حاسوب محمول","notebook computer","لابتوب"], "mantel":["mantel","مدفأة","fireplace mantel","رف مدفأة"], "monitor":["monitor","شاشة","computer screen","شاشة حاسوب"], "night_stand":["night stand","كومودينو","bedside table","طاولة سرير"], "person":["person","شخص","human figure","انسان"], "piano":["piano","بيانو","grand piano","بيانو كبير"], "plant":["plant","نبات","green plant","نبتة"], "radio":["radio","مذياع","old radio","راديو"], "range_hood":["range hood","شفاط مطبخ","kitchen hood","غطاء مطبخ"], "sink":["sink","مغسلة","wash basin","حوض غسل"], "sofa":["sofa","أريكة","three seat sofa","كنبة"], "stairs":["stairs","درج","staircase","سلالم"], "stool":["stool","تابور","bar stool","كرسي بلا ظهر"], "table":["table","طاولة","dining table","طاولة خشب"], "tent":["tent","خيمة","camping tent","خيمة كبيرة"], "toilet":["toilet","مرحاض","bathroom toilet","حمام"], "tv_stand":["tv stand","طاولة تلفاز","television stand","حامل تلفاز"], "vase":["vase","مزهرية","ceramic vase","مزهرية زهور"], "wardrobe":["wardrobe","دولاب","closet wardrobe","خزانة ملابس"], "xbox":["xbox","جهاز العاب","game console","صندوق العاب"],}device = "cuda" if torch.cuda.is_available() else "cpu"model = Gen().to(device)print("البارامترات:", f"{sum(p.numel() for p in model.parameters()):,}")if torch.cuda.device_count() > 1:    model = nn.DataParallel(model)    print("كرتين:", torch.cuda.device_count())

In [ ]:
# ٦) التدريبimport torch, torch.nn as nn, time, randomfrom torch.utils.data import Dataset, DataLoaderclass VoxDS(Dataset):    def __init__(self, occs, cls_names, variant=3, train=True):        self.items = []        for o, c in zip(occs, cls_names):            for nm in NAMES.get(c, [c])[:variant]:                self.items.append((nm, o))        self.train = train    def __len__(self): return len(self.items)    def __getitem__(self, i):        nm, o = self.items[i]        x, y = text_feat(nm), o.astype(np.float32)        if self.train and random.random() < 0.4:            y = y[::-1].copy()        return torch.from_numpy(x), torch.from_numpy(y)EPOCHS, BATCH = 40, 16ds = VoxDS(occs, cls_names, 3, True)dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)print("عيّنات تدريب:", len(ds))opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS*len(dl))pos_w = torch.tensor(float(1/max(occs.mean(), 1e-3) - 1)).to(device)crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)print("وزن الموجب:", round(float(pos_w), 1))t0 = time.time()for ep in range(EPOCHS):    model.train(); tot, n = 0, 0    for x, y in dl:        x, y = x.to(device), y.to(device)        loss = crit(model(x), y)        opt.zero_grad(); loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        opt.step(); sched.step()        tot += loss.item(); n += 1    print(f"epoch {ep+1:2d}/{EPOCHS} · loss {tot/n:.4f} · {(time.time()-t0)/60:.1f} دقيقة", flush=True)OUT = "/content" if os.path.isdir("/content") else "."torch.save(model.state_dict(), OUT + "/ayso3d_state.pt")print("حُفظت الأوزان ✓")

In [ ]:
# ٧) التقييمmodel.eval()def predict(text, thr=0.5):    with torch.no_grad():        x = torch.from_numpy(text_feat(text)).unsqueeze(0).to(device)        p = torch.sigmoid(model(x))[0].cpu().numpy()    return (p > thr), pdef iou(a, b):    i = np.logical_and(a, b).sum(); u = np.logical_or(a, b).sum()    return i / u if u else 0.0s, n = 0, 0for o, c in zip(occs[:80], cls_names[:80]):    s += iou(predict(NAMES.get(c, [c])[0])[0], o); n += 1print(f"IoU على نصوص شُوفت      : {s/n*100:.1f}%")s, n = 0, 0for o, c in zip(occs, cls_names):    nm = (NAMES.get(c, [c]) + [c])[3]    s += iou(predict(nm)[0], o); n += 1print(f"IoU على نصوص جديدة      : {s/n*100:.1f}%")avg = {}for c in set(cls_names):    idx = [i for i, x in enumerate(cls_names) if x == c]    avg[c] = occs[idx].mean(0) > 0.5s = sum(iou(avg[c], o) for o, c in zip(occs, cls_names))print(f"سقف الشكل المتوسط       : {s/len(occs)*100:.1f}%")print(f"خط الأساس (فاضي)        : {(1-occs.mean())*100:.1f}%")

In [ ]:
# ٨) عرض بصريimport matplotlib.pyplot as pltfig, axs = plt.subplots(2, 4, figsize=(16, 8))for i, t in enumerate(["chair", "طاولة", "سيارة", "مصباح"]):    occ, prob = predict(t)    axs[0, i].imshow(np.flipud(prob.max(axis=2).T), cmap='viridis'); axs[0, i].set_title(t); axs[0, i].axis('off')    axs[1, i].imshow(np.flipud(occ.max(axis=2).T), cmap='gray_r'); axs[1, i].axis('off')plt.tight_layout()plt.savefig(OUT + "/samples.png", dpi=110)plt.show()

In [ ]:
# ٩) تصدير ONNX + مجسّمات GLB + حزمةimport os, torch, numpy as npOUT = "/content" if os.path.isdir("/content") else "."m = model.module if hasattr(model, 'module') else modelm.eval()torch.onnx.export(m, torch.zeros(1, D_TXT).to(device), OUT + "/ayso3d.onnx",                  input_names=["text"], output_names=["logits"],                  dynamic_axes={"text": {0: "batch"}, "logits": {0: "batch"}},                  opset_version=17)print("ONNX:", round(os.path.getsize(OUT + '/ayso3d.onnx')/1048576, 2), "ميجا")from skimage import measureimport trimeshfor t in ["chair", "table", "car", "lamp"]:    occ, prob = predict(t)    try:        v, f, _, _ = measure.marching_cubes(prob.astype(np.float32), level=0.5)        mesh = trimesh.Trimesh(vertices=v, faces=f, process=True)        mesh.apply_scale(1.0 / 32.0)        mesh.export(f"{OUT}/{t}.glb")        print(f"{t}.glb ✓ {len(mesh.faces)} مثلث")    except Exception as e:        print(t, "فشل:", str(e)[:80])os.system(f"cd {OUT} && zip -q ayso3d-model.zip ayso3d_state.pt ayso3d.onnx *.glb samples.png 2>/dev/null")from google.colab import files as _f_f.download(OUT + "/ayso3d-model.zip")print("نُزّل ✓")